In [1]:
import matplotlib.pyplot as plt
import matplotlib
import csv
import numpy as np
import pandas as pd
import os
import re
import scipy.stats as ss
import rmgpy.constants
from rmgpy.reaction import Reaction
from rmgpy.species import Species
from rmgpy.molecule import Molecule
from rmgpy.kinetics.arrhenius import Arrhenius
from rmgpy.data.rmg import RMGDatabase
from rmgpy import settings
from rmgpy.chemkin import load_chemkin_file
from rmgpy.rmg.pdep import PDepReaction
from arkane.input import load_input_file
from IPython.display import IFrame
import matplotlib.lines as mlines

%matplotlib inline

# Thermo

In [2]:
#master list for thermo
master_dictionary_thermo = {}

In [3]:
new_families = [                     #new families
                     'CO2_Elimination_From_Carboxylic_Acid',
                     'CO_CF_bond_dissociation', 
                     'Enol_Ether_Formation', 
                     'Lactone_Formation',
                     'PFAS_Hydrolysis',
                     'Perfluoroalkene_Formation'
]

In [4]:
thermolibs = [
    'NCSU_2025_revised',
    'C1_C2_Fluorine', #putting Siddha's as most trusted because Caroline used this thermo for calcs
    'PFCA_thermo',
    # 'Fluorine',
    # 'primaryThermoLibrary',
    # 'FFCM1(-)',
    # 'halogens',
    # 'CHOF_G4',
    # 'CHOCl_G4',
    # 'CHOBr_G4',
    # 'CHOFCl_G4',
    # 'CHOFBr_G4',
    # 'CHOFClBr_G4',
    # 'DFT_QCI_thermo',
    # '2-BTP_G4',
    # 'thermo_DFT_CCSDTF12_BAC',
    # 'SulfurHaynes'
    ]

families = new_families
database = RMGDatabase()
database.load(
    path = settings['database.directory'],
    thermo_libraries = thermolibs,  # Can add others if necessary
    kinetics_families = families,
    reaction_libraries = ['PFOA_unimolecular_decomp', "ANL_Brown_pdep", "ANL_Brown_pdep_2", 'Brown_difluoromethane_seed'],
    kinetics_depositories = ['training'],
)

In [46]:
library.get_entries_to_save()[0].data

NASA(polynomials=[NASAPolynomial(coeffs=[-962.551,24.8316,3.2682,0.00103959,-2.2605e-06,2.27932e-09,-7.21878e-13,-33951,2.12947], Tmin=(100,'K'), Tmax=(1000,'K')), NASAPolynomial(coeffs=[725509,-1485.62,3.85873,0.00071085,-2.09543e-07,3.03112e-11,-1.63173e-15,-23482.3,-3.2241], Tmin=(1000,'K'), Tmax=(6000,'K'))], Tmin=(100,'K'), Tmax=(6000,'K'))

In [35]:
for lib_name, library in database.thermo.libraries.items(): 

    for species_name, entry in library.entries.items():

        
        NASA_poly = entry.data


        



    

In [77]:
str(list(NASA_poly.polynomials[0].coeffs))

'[-7.82610708, 0.292294164, -0.000540101631, 4.86542445e-07, -1.66855684e-10, -375127.573, 71.0913699]'

In [79]:
[float(x) for x in lower_temps_NASA.coeffs]

[-7.82610708,
 0.292294164,
 -0.000540101631,
 4.86542445e-07,
 -1.66855684e-10,
 -375127.573,
 71.0913699]

In [81]:
import yaml
import os


for lib_name, library in database.thermo.libraries.items():
    library_data = []

    for species_name, entry in library.entries.items():
        
        NASA_poly = entry.data

        lower_temps_NASA = NASA_poly.polynomials[0]
        higher_temps_NASA = NASA_poly.polynomials[1]


        if higher_temps_NASA.Tmin.value == higher_temps_NASA.Tmax.value:
            continue
        
        assert lower_temps_NASA.Tmax.value==higher_temps_NASA.Tmin.value

        species_entry = {
            "name": species_name,
            "thermo": {"model": 'NASA', 
                       "temperature-ranges": [lower_temps_NASA.Tmin.value, lower_temps_NASA.Tmax.value, higher_temps_NASA.Tmax.value], 
                       "data": [[float(x) for x in lower_temps_NASA.coeffs], [float(x) for x in higher_temps_NASA.coeffs]]
                        }
        }

        library_data.append(species_entry)

    yaml_output = yaml.dump(library_data, sort_keys=False, default_flow_style=None)
    print(yaml_output)

    with open(f"./SI/yamls_for_thermo/{lib_name}_library_thermo.yaml", "w") as f:
        f.write(yaml_output)

- name: O
  thermo:
    model: NASA
    temperature-ranges: [200.0, 1000.0, 6000.0]
    data:
    - [3.1682671, -0.00327931884, 6.64306396e-06, -6.12806624e-09, 2.11265971e-12,
      29122.2592, 2.05193346]
    - [2.54363697, -2.73162486e-05, -4.1902952e-09, 4.95481845e-12, -4.79553694e-16,
      29226.012, 4.92229457]
- name: F
  thermo:
    model: NASA
    temperature-ranges: [200.0, 1000.0, 6000.0]
    data:
    - [2.4196743, 0.0029392909, -8.9212228e-06, 9.9118537e-09, -3.7947152e-12, 8757.322,
      4.7468987]
    - [2.6716339, -0.00017461853, 6.9066504e-08, -1.1953478e-11, 7.5236739e-16, 8787.4123,
      3.9842568]
- name: H2
  thermo:
    model: NASA
    temperature-ranges: [200.0, 1000.0, 6000.0]
    data:
    - [2.34433112, 0.00798052075, -1.9478151e-05, 2.01572094e-08, -7.37611761e-12,
      -917.935173, 0.683010238]
    - [2.93286575, 0.000826608026, -1.46402364e-07, 1.54100414e-11, -6.888048e-16,
      -813.065581, -1.02432865]
- name: O2
  thermo:
    model: NASA
    tempe

In [68]:
species_name

'H'

In [67]:
lib_name

'NCSU_2025_revised'

In [66]:
NASA_poly

NASA(polynomials=[NASAPolynomial(coeffs=[2.5,0,0,0,0,25473.7,-0.446683], Tmin=(200,'K'), Tmax=(6000,'K')), NASAPolynomial(coeffs=[2.5,0,0,0,0,25473.7,-0.446683], Tmin=(6000,'K'), Tmax=(6000,'K'))], Tmin=(200,'K'), Tmax=(6000,'K'))

In [62]:
lower_temps_NASA.Tmax.value

6000.0

In [64]:
lower_temps_NASA.Tmin.value

200.0

In [65]:
higher_temps_NASA.Tmax.value

6000.0

In [63]:
higher_temps_NASA.Tmin.value

200.0

In [34]:
def format_num(x, sig=4):
    """Format a number in scientific/clean notation for LaTeX."""
    if x is None:
        return "--"
    return f"{x:.{sig}g}"

latex_rows = []
temps = list(range(300, 2001, 100))  # fixed, same for all species
n_temps = len(temps)

for lib_name, library in database.thermo.libraries.items():
    for species_name, entry in library.entries.items():
        H298 = entry.data.get_enthalpy(298)
        S298 = entry.data.get_entropy(298)
        Cps = [entry.data.get_heat_capacity(T) for T in temps]
        # smiles = entry.item.smiles

        species_escaped = species_name.replace("_", r"\_")
        # smiles_escaped = smiles.replace("_", r"\_")

        cp_values = " & ".join(format_num(cp) for cp in Cps)

        row = (
            f"{species_escaped} & "
            # f"{smiles_escaped} & "
            f"{format_num(H298)} & "
            f"{format_num(S298)} & "
            f"{cp_values} \\\\[10pt]"
        )
        latex_rows.append(row)

    n_fixed_cols = 3  # Species H298, S298
    n_cols = n_fixed_cols + n_temps
    col_spec = "p{2.5cm}p{1.5cm}p{1.5cm}" + ">{\scriptsize}c" * n_temps
    
    # Header rows
    temp_header_row = " & ".join(str(T) for T in temps)
    fixed_header_blank = " & " * n_fixed_cols  # empty cells above Species/SMILES/H298/S298
    
    header_block = (
        r"Species & $H_{298}$ & $S_{298}$ & "
        r"\multicolumn{" + str(n_temps) + r"}{c}{$C_p(T)$ [J/mol$\cdot$K]} \\" "\n"
        r"\cline{" + str(n_fixed_cols + 1) + "-" + str(n_cols) + "}\n"
        r" & & [J/mol] & [J/mol$\cdot$K] & " + temp_header_row + r" \\" "\n"
    )

    latex_table = (
        r"\begin{footnotesize}" "\n"
        r"\begin{longtable}{" + col_spec + "}\n"
        r"\caption{Thermodynamic parameters for library species.}\\" "\n"
        r"\hline" "\n"
        + header_block +
        r"\hline" "\n"
        r"\endfirsthead" "\n"
        r"\multicolumn{" + str(n_cols) + r"}{c}{\tablename\ \thetable{} -- continued from previous page} \\" "\n"
        r"\hline" "\n"
        + header_block +
        r"\hline" "\n"
        r"\endhead" "\n"
        r"\hline \multicolumn{" + str(n_cols) + r"}{r}{Continued on next page} \\" "\n"
        r"\endfoot" "\n"
        r"\hline" "\n"
        r"\multicolumn{" + str(n_cols) + r"}{p{16cm}}{\small\textit{Note:} $H_{298}$ and $S_{298}$ are the "
        r"standard enthalpy and entropy at 298~K, in J/mol and J/mol$\cdot$K respectively; "
        r"$C_p(T)$ is tabulated at $T = 300$--$2000$~K in 100~K increments, units J/mol$\cdot$K.} \\" "\n"
        r"\endlastfoot" "\n"
        + (r"\hline" "\n").join(latex_rows) +
        "\n" r"\end{longtable}" "\n"
        r"\end{footnotesize}"
    )

    
    
    print(f'%%%%%%%%%%%%%%%%%%%{lib_name}%%%%%%%%%%%%%%%%%%%%%%%')
    print(latex_table)

%%%%%%%%%%%%%%%%%%%NCSU_2025_revised%%%%%%%%%%%%%%%%%%%%%%%
\begin{footnotesize}
\begin{longtable}{p{2.5cm}p{1.5cm}p{1.5cm}>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c>{\scriptsize}c}
\caption{Thermodynamic parameters for library species.}\\
\hline
Species & $H_{298}$ & $S_{298}$ & \multicolumn{18}{c}{$C_p(T)$ [J/mol$\cdot$K]} \\
\cline{4-21}
 & & [J/mol] & [J/mol$\cdot$K] & 300 & 400 & 500 & 600 & 700 & 800 & 900 & 1000 & 1100 & 1200 & 1300 & 1400 & 1500 & 1600 & 1700 & 1800 & 1900 & 2000 \\
\hline
\endfirsthead
\multicolumn{21}{c}{\tablename\ \thetable{} -- continued from previous page} \\
\hline
Species & $H_{298}$ & $S_{298}$ & \multicolumn{18}{c}{$C_p(T)$ [J/mol$\cdot$K]} \\
\cline{4-21}
 & & [J/mol] & [J/mol$\cdot$K] & 300 & 400 & 500 & 600 & 700 & 800 & 900 & 1000 & 1100 

SyntaxError: f-string expression part cannot include a backslash (4230020827.py, line 1)

In [17]:
library.entries['HF'].data.get_enthalpy(298)

#'get_enthalpy', #Return the enthalpy in J/mol for the species at the specified temperature T in K.
# 'get_entropy', #Return the entropy in J/mol*K for the species at the specified temperature T in K.
# 'get_free_energy', 
# 'get_heat_capacity #Return the heat capacity in J/mol*K for the species at the specified temperature T in K.

-272731.3703361692

In [ ]:
family_of_interest = [
                      '1+2_Cycloaddition',            #30 found 
                     '1,2_Insertion_CO',             #26 found 
                     '1,3_Insertion_CO2',            #15 found 
                     '1,3_sigmatropic_rearrangement',  #8 found
                     'F_Abstraction',                  #43 found
                     'H_Abstraction',                  #687 found
                     'R_Addition_COm',                #10 found 
                     'R_Addition_MultipleBond',       #58 found 
                     'XY_Addition_MultipleBond',       #33 found
                     '1,2_Insertion_carbene',         # 51 found 
                     'CO_Disproportionation',           #20 found
                     'Disproportionation',              #124 found
                     'Singlet_Carbene_Intra_Disproportionation', #4 found 
                     'halocarbene_recombination',      #4 individually found 
                     'halocarbene_CO_dimerization',                   
                     


                     'CO2_Elimination_From_Carboxylic_Acid',
                     'CO_CF_bond_dissociation', 
                     'Enol_Ether_Formation', 
                     'Lactone_Formation',
                     'PFAS_Hydrolysis',
                     'Perfluoroalkene_Formation'

    
]

for fam in family_of_interest: 

    training_depo = database.kinetics.families[fam].get_training_depository()
    collected_reactions_for_this_family = []
    
    if fam not in new_families: 
        count = 0 
        if fam == 'halocarbene_recombination':
            kphrases = halocarbene_recombination_key_phrases
        elif fam == 'R_Addition_MultipleBond': 
            kphrases = ramb_key_phrases
        elif fam == 'XY_Addition_MultipleBond': 
            kphrases = xyamb_key_phrases
        else: 
            kphrases = key_phrases
            
    
        if fam == 'Disproportionation': 
            kphrases = disproportionation_key_phrases
            for tr in training_depo.entries.values():   
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    #print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1

                #test in short description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.short_desc), None)
                if matched_phrase:
                    # print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1
    
        elif fam == 'H_Abstraction': 
            kphrases = habstraction_key_phrases 
            for tr in training_depo.entries.values(): 

                #test in long description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    # print(tr.long_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1
                #test in short description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.short_desc), None)
                if matched_phrase:
                    # print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1
        
                    
        elif fam == 'CO_Disproportionation': 
            kphrases = CO_disp_key_phrases
            for tr in training_depo.entries.values(): 
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    #print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1

                #test in short description
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.short_desc), None)
                if matched_phrase:
                    # print(tr.short_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1

        else: 
            for tr in training_depo.entries.values(): 
                matched_phrase = next((key_phrase for key_phrase in kphrases if key_phrase in tr.long_desc), None)
                if matched_phrase:
                    # print(tr.long_desc)
                    collected_reactions_for_this_family.append([tr, kphrases[matched_phrase]])
                    count+=1


                    
    if fam in new_families: 
        count = 0 
        for tr in training_depo.entries.values(): 
            collected_reactions_for_this_family.append([tr, PW])
            count+=1

    master_dictionary_kinetics[fam] = collected_reactions_for_this_family
    assert(len(collected_reactions_for_this_family)==count)
    print(f'{fam}: {count}')
    
assert expecting_dictionary=={k:len(v) for k,v in master_dictionary_kinetics.items()}

In [ ]:
print(training_reaction.data.A)

In [ ]:
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        print(fam, training_reaction, training_reaction.data.A, training_reaction.data.n, training_reaction.data.Ea, source)
        

In [ ]:
#latex code for formatting rate data 
def format_A(value):
    # Use exponential notation for very large or very small numbers, 
    # otherwise use fixed 2 decimal places
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"


latex_rows = []
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        rxn_str = str(training_reaction).replace('_', r'\_')
        fam_str = str(fam).replace('_', r'\_')
        source_str = str(source).replace('_', r'\_')

        # A = str(training_reaction.data.A).replace(' ', '\:').replace(f'{training_reaction.data.A.value}', f'{training_reaction.data.A.value:.2f}')
        A_value_str = format_A(training_reaction.data.A.value)
        A = f'{A_value_str} ${training_reaction.data.A.units}$'
        n = training_reaction.data.n.value_si
        Ea = training_reaction.data.Ea.value_si
        # rate_data = f"$A$ = ${A}$, $n$ = {n}, $E_a$ = {Ea:.3e}"

        latex_rows.append(f"{rxn_str} & {A} & {n:.2f} & {Ea:.3e} & {fam_str} & {source_str} \\\\")

latex_table = (
    "\\begin{footnotesize}\n"
    "\\begin{longtable}{p{5cm}p{2cm}p{1.3cm}p{2cm}p{2.5cm}p{2cm}}\n"
    "\\caption{Kinetic parameters for training reactions (SI units).} \\label{tab:training_reactions} \\\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
    "\\hline\n"
    "\\endfirsthead\n"
    "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
    "\\hline\n"
    "\\endhead\n"
    "\\hline \\multicolumn{6}{r}{{Continued on next page}} \\\\\n"
    "\\endfoot\n"
    "\\hline\n"
    "\\endlastfoot\n"
    + "\n".join(latex_rows) +
    "\n\\end{longtable}\n"
    "\\end{footnotesize}"
)

print(latex_table)

In [ ]:
family_of_interest = [
                      '1+2_Cycloaddition',            #30 found 
                     # '1,2_Insertion_CO',             #26 found 
                     # '1,3_Insertion_CO2',            #15 found 
                     # '1,3_sigmatropic_rearrangement',  #8 found
                     # 'F_Abstraction',                  #43 found
                     # 'H_Abstraction',                  #687 found
                     # 'R_Addition_COm',                #10 found 
                     # 'R_Addition_MultipleBond',       #58 found 
                     # 'XY_Addition_MultipleBond',       #33 found
                     # '1,2_Insertion_carbene',         # 51 found 
                     # 'CO_Disproportionation',           #20 found
                     # 'Disproportionation',              #124 found
                     # 'Singlet_Carbene_Intra_Disproportionation', #4 found 
                     # 'halocarbene_recombination',      #4 individually found 
                     # 'halocarbene_CO_dimerization',                   
                     


                     # 'CO2_Elimination_From_Carboxylic_Acid',
                     # 'CO_CF_bond_dissociation', 
                     # 'Enol_Ether_Formation', 
                     # 'Lactone_Formation',
                     # 'PFAS_Hydrolysis',
                     # 'Perfluoroalkene_Formation'

    
]

In [ ]:
#latex code for formatting rate data 
def format_A(value):
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"

# Build a mapping of unique sources (excluding "present work") to reference numbers
source_to_number = {}
next_ref_number = 1
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        source_str = str(source)
        if source_str != "present work" and source_str not in source_to_number:
            source_to_number[source_str] = next_ref_number
            next_ref_number += 1

latex_rows = []
for fam in family_of_interest:
    collected_rxn_data = master_dictionary_kinetics[fam]
    for [training_reaction, source] in collected_rxn_data:
        rxn_str = str(training_reaction).replace('_', r'\_')
        fam_str = str(fam).replace('_', r'\_')

        source_str = str(source)
        if source_str == "present work":
            source_display = "PW"
        else:
            source_display = f"[{source_to_number[source_str]}]"

        A_value_str = format_A(training_reaction.data.A.value)
        A = f'{A_value_str}'
        n = training_reaction.data.n.value_si
        Ea = training_reaction.data.Ea.value_si
        latex_rows.append(f"{rxn_str} & {A} & {n:.2f} & {Ea:.3e} & {source_display} \\\\")

footnote_text = (
    "The units of the $A$ factor depend on the order of the reaction: "
    "$s^{-1}$ for unimolecular (first-order) reactions, "
    "$cm^3/(mol \\cdot s)$ for bimolecular (second-order) reactions, "
    "and $cm^6/(mol^2 \\cdot s)$ for termolecular (third-order) reactions. "
    "PW stands for present work."
)

latex_table = (
    "\\begin{footnotesize}\n"
    "\\begin{longtable}{p{7cm}p{1.5cm}p{1.3cm}p{1.5cm}p{0.5cm}>{\centering\arraybackslash}p{2cm}}\n"
    "\caption{FAM Training reactions.}\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\\\\n"
    "\\hline\n"
    "\\endfirsthead\n"
    "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
    "\\hline\n"
    "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Source \\\\\n"
    "\hline\n"
    "\endhead\n"
    "\hline \multicolumn{6}{r}{{Continued on next page}} \\\n"
    "\endfoot\n"
    "\hline\n"
    "\multicolumn{6}{p{14cm}}{\small\textit{Note:} Units of $A$ depend on reaction order: $s^{-1}$ (unimolecular), $cm^3/(mol \cdot s)$ (bimolecular), $cm^6/(mol^2 \cdot s)$ (termolecular). PW = present work.} \\\n"
    "\endlastfoot\n"
    + "\n".join(latex_rows) +
    "\n\\end{longtable}\n"
    "\\end{footnotesize}"
)
print(latex_table)

# Print the reference key so you know which number maps to which source
print("\n% Reference key:")
for src, num in source_to_number.items():
    print(f"% [{num}] {src}")

In [ ]:
#latex code for formatting rate data 
def format_A(value):
    if value == 0:
        return "0.00"
    elif abs(value) >= 1e4 or abs(value) < 1e-2:
        return f"{value:.2e}"
    else:
        return f"{value:.2f}"

# Build a mapping of unique sources (excluding "present work") to reference numbers
source_to_number = {}
next_ref_number = 1
for fam, collected_rxn_data in master_dictionary_kinetics.items():
    for [training_reaction, source] in collected_rxn_data:
        source_str = str(source)
        if source_str != "present work" and source_str not in source_to_number:
            source_to_number[source_str] = next_ref_number
            next_ref_number += 1


for fam, collected_rxn_data in master_dictionary_kinetics.items():

    latex_rows = []
    
    for [training_reaction, source] in collected_rxn_data:
        rxn_str = str(training_reaction).replace('_', r'\_')
        fam_str = str(fam).replace('_', r'\_')

        source_str = str(source)
        if source_str == "present work":
            source_display = "PW"
        else:
            source_display = f"[{source_to_number[source_str]}]"

        A_value_str = format_A(training_reaction.data.A.value)
        A = f'{A_value_str}'
        n = training_reaction.data.n.value_si
        Ea = training_reaction.data.Ea.value_si
        latex_rows.append(f"{rxn_str} & {A} & {n:.2f} & {Ea:.3e} & {fam_str} & {source_display} \\\\")

    footnote_text = (
        "The units of the $A$ factor depend on the order of the reaction: "
        "$s^{-1}$ for unimolecular (first-order) reactions, "
        "$cm^3/(mol \\cdot s)$ for bimolecular (second-order) reactions, "
        "and $cm^6/(mol^2 \\cdot s)$ for termolecular (third-order) reactions. "
        "PW stands for present work."
    )
    
    latex_table = (
        "\\begin{footnotesize}\n"
        "\\begin{longtable}{p{5cm}p{2cm}p{1.3cm}p{2cm}p{2.5cm}>{\\centering\\arraybackslash}p{2cm}}\n"
        f"\\caption{{Kinetic parameters for training reactions (SI units).\\footnote{{{footnote_text}}}}} \\label{{tab:training_reactions}} \\\\\n"
        "\\hline\n"
        "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
        "\\hline\n"
        "\\endfirsthead\n"
        "\\multicolumn{6}{c}{\\tablename\\ \\thetable{} -- continued from previous page} \\\\\n"
        "\\hline\n"
        "Reaction & $A$ & $n$ & $E_a$ [J/mol] & Family & Source \\\\\n"
        "\\hline\n"
        "\\endhead\n"
        "\\hline \\multicolumn{6}{r}{{Continued on next page}} \\\\\n"
        "\\endfoot\n"
        "\\hline\n"
        "\\endlastfoot\n"
        + "\n".join(latex_rows) +
        "\n\\end{longtable}\n"
        "\\end{footnotesize}"
    )
    print(latex_table)

    # Print the reference key so you know which number maps to which source
    print("\n% Reference key:")
    for src, num in source_to_number.items():
        print(f"% [{num}] {src}")